# PHQ-8 GP-vs-EN follow-up: structure, criteria sensitivity, effect size, exhaustive search

Follow-up to the finding that the full PHQ-8 fails **configural** invariance between the general-population and enriched samples (gp_en, grid1st data; see `data/cfa_other_gp_en/`). Four analyses:

- **A. Conventional-criteria sensitivity screen** (fits only, no permutations): the same measEq ladder evaluated with the field-typical ΔCFI ≤ .01 (Cheung & Rensvold 2002) and ΔRMSEA ≤ .015 (Chen 2007) change criteria, side by side with the exact permutation verdicts — positions our stricter test against the literature. Run for all six non-HiTOP scales for context.
- **B. Per-group EFA of the PHQ-8**: polychoric eigenvalues, 1- and 2-factor geomin EFAs, and where the 1-factor residuals concentrate in each group — the direct diagnostic of *what* differs structurally when configural fails.
- **C. Effect size of the non-invariance**: per-item signed expected-score bias and dMACS (Nye & Drasgow 2011), plus the expected **sum-score bias at equal latent severity** — the practically interpretable number for the "pooled brain–PHQ correlation" concern.
- **D. Exhaustive scalar-target search** for a PHQ-8 invariant subset on gp_en (permutation-based; resumable; capped at size 6 by the stepwise run's configural certificate).

Sections A–C are cheap (model fits only). Section D is the long one. Outputs in `data/cfa_phq8_followup/`. Run headless with `./notebooks/run_overnight.sh NB_phq8_gp_en_followup.ipynb`.

In [ ]:
import os
for _var in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_var, "1")
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

import hitop_cfa
from hitop_cfa.r_env import ro, localconverter, pandas2ri
from hitop_cfa import (WU_ESTABROOK_LEVELS, fit_measeq_level, set_seeds,
                       exhaustive_search_scale, stepwise_configural_cap)
from hitop_cfa.measeq import fit_is_converged
from hitop_cfa.fit import push_model_to_r
from hitop_cfa.scales import OTHER_SCALES

In [ ]:
log_dir = Path('./log'); log_dir.mkdir(exist_ok=True)
dat_dir = Path('../data/')
fin_dir = dat_dir / 'finaldata'
cfa_dir = dat_dir / 'cfa_phq8_followup'
cfa_dir.mkdir(exist_ok=True)
helpfile_dir = cfa_dir / 'temp'; helpfile_dir.mkdir(exist_ok=True, parents=True)
path_to_helpfile = helpfile_dir / 'cfa_temp.csv'

In [ ]:
def get_architecture():
    arch = platform.machine().lower()
    if "arm" in arch or "aarch" in arch:
        return "ARM"
    return "x86"

total_cpus = os.cpu_count()
cpus_to_use = total_cpus - 2 if get_architecture() == 'ARM' else total_cpus // 2 - 1
print(f"Using {cpus_to_use} CPUs for permutation tests")

num_iter = 1000
exh_min_items = 3   # section D floor (sandbox overrides may raise this)

In [ ]:
set_seeds(12345)

In [ ]:
# grid1st data, concat order identical to cfa_other_gp_en (gp first):
# permutation draws depend on row order, so this keeps every test here --
# and the stepwise configural certificate used in section D -- seed-valid
data_gp = pd.read_csv(fin_dir / 'dat_gp_grid1st_full.csv')
data_en = pd.read_csv(fin_dir / 'dat_en_grid1st_full.csv')
data_genpop_enriched = pd.concat([data_gp, data_en])

PHQ_FORMULA = OTHER_SCALES['phq_sum']
PHQ_ITEMS = PHQ_FORMULA.split('=~', 1)[1].strip().split(' + ')
print(f"gp n={len(data_gp)}, en n={len(data_en)}; PHQ items: {PHQ_ITEMS}")

## A. Conventional-criteria sensitivity screen

Same Wu–Estabrook ladder, judged the way most published invariance studies judge it: change in scaled CFI/RMSEA (the robust variants are undefined for the constrained measEq fits; scaled variants are what the WLSMV invariance literature uses) between consecutive levels (pass: ΔCFI ≥ −.010 and ΔRMSEA ≤ +.015), plus the configural model's absolute fit. No permutations — this is purely about whether the field's usual yardstick would have flagged what the exact permutation test flagged. (Note: some authors argue even stricter Δ cutoffs for ordinal WLSMV; .01/.015 is the common practice being emulated.)

In [ ]:
lax_rows = []
for scale, formula in OTHER_SCALES.items():
    items = formula.split('=~', 1)[1].strip().split(' + ')
    push_model_to_r(scale, items, data_genpop_enriched, path_to_helpfile)
    prev = None
    for level, ge in WU_ESTABROOK_LEVELS:
        fit_measeq_level(items, ge, r_fit_name=f'fit_{level}')
        if not fit_is_converged(f'fit_{level}'):
            lax_rows.append(dict(scale=scale, level=level, converged=False))
            break
        cfi, rmsea, chisq, df = (float(v) for v in ro.r(
            f'lavaan::fitMeasures(fit_{level}, '
            f'c("cfi.scaled", "rmsea.scaled", "chisq.scaled", "df"))'))
        row = dict(scale=scale, level=level, converged=True,
                   cfi=round(cfi, 4), rmsea=round(rmsea, 4),
                   chisq_scaled=round(chisq, 2), df=int(df))
        if prev is not None and pd.notna(cfi) and pd.notna(prev['cfi']):
            row['delta_cfi'] = round(cfi - prev['cfi'], 4)
            row['delta_rmsea'] = round(rmsea - prev['rmsea'], 4)
            row['passes_dCFI'] = row['delta_cfi'] >= -0.010
            row['passes_dRMSEA'] = row['delta_rmsea'] <= 0.015
        lax_rows.append(row)
        prev = row
lax = pd.DataFrame(lax_rows)
lax.to_csv(cfa_dir / 'laxer_criteria_screen.csv', index=False)
lax

In [ ]:
# side-by-side: what the exact permutation test said (cfa_other_gp_en run)
perm_path = dat_dir / 'cfa_other_gp_en' / 'orig_cfa_res.csv'
if perm_path.exists():
    perm = pd.read_csv(perm_path)
    print("exact permutation baseline (p >= .05 passes; NaN = level never reached):")
    print(perm[['scale', 'pconfig', 'pthresholds', 'pmetric', 'pscalar',
                'pstrict']].to_string(index=False))
else:
    print(f"{perm_path} not found -- run NB_2_cfa_as_reg_other_gp_en first")

## B. Per-group EFA of the PHQ-8

Configural failure means the one-factor model does not fit equivalently in the two groups; this section shows what each group's structure looks like: polychoric eigenvalues (scree), the 1-factor solution's loadings and worst residual correlations (where the misfit lives), and a 2-factor geomin EFA.

In [ ]:
for label, df in (('genpop', data_gp), ('enriched', data_en)):
    print(f"\n================ {label} (n={len(df)}) ================")
    df[PHQ_ITEMS].to_csv(path_to_helpfile, index=False)
    ro.r(f'gdat <- read.csv("{path_to_helpfile}")')
    ro.r('for (cc in names(gdat)) gdat[[cc]] <- ordered(gdat[[cc]])')
    ro.r('pc <- lavaan::lavCor(gdat)')
    eig = np.array(ro.r('eigen(pc)$values'))
    print('polychoric eigenvalues:', np.round(eig, 3))

    ro.globalenv['model1'] = 'phq =~ ' + ' + '.join(PHQ_ITEMS)
    ro.r('f1 <- lavaan::cfa(model1, data = gdat, ordered = names(gdat), '
         'estimator = "WLSMV", parameterization = "theta", std.lv = TRUE)')
    print('\n1-factor fit:')
    ro.r('print(round(lavaan::fitMeasures(f1, c("chisq.scaled", "df", '
         '"pvalue.scaled", "cfi.robust", "rmsea.robust")), 3))')
    print('1-factor standardized loadings:')
    ro.r('print(round(lavaan::inspect(f1, "std")$lambda, 2))')
    print('largest residual polychoric correlations (1-factor):')
    ro.r('res <- lavaan::lavResiduals(f1)$cov; res[upper.tri(res, diag = TRUE)] <- NA')
    ro.r('ix <- order(abs(res), decreasing = TRUE, na.last = NA)[1:5]; '
         'rn <- rownames(res)[row(res)[ix]]; cn <- colnames(res)[col(res)[ix]]; '
         'print(data.frame(item1 = rn, item2 = cn, resid = round(res[ix], 3)))')

    ro.globalenv['model2'] = ('efa("b1")*f1 + efa("b1")*f2 =~ '
                              + ' + '.join(PHQ_ITEMS))
    ro.r('e2 <- lavaan::cfa(model2, data = gdat, ordered = names(gdat), '
         'estimator = "WLSMV", parameterization = "theta", std.lv = TRUE, '
         'rotation = "geomin")')
    print('\n2-factor geomin EFA loadings:')
    ro.r('print(round(lavaan::inspect(e2, "std")$lambda, 2))')
    ro.r('print(round(lavaan::fitMeasures(e2, c("chisq.scaled", "df", '
         '"pvalue.scaled", "cfi.robust", "rmsea.robust")), 3))')

## C. Effect size of the non-invariance (DIF beyond a true severity difference)

The naive comparison of configural-model curves conflates real severity differences with measurement bias (std.lv standardizes η within each group). The correct isolation:

- fit the **metric** model (common loadings λ and thresholds τ; group-2 item intercepts ν̂ free) — ν̂ᵢ absorbs both a genuine latent-mean difference and item-specific DIF;
- fit the **scalar** model (all ν = 0, group-2 latent mean α̂ free) — α̂ is the severity difference a fully invariant instrument would report;
- per-item **non-uniform DIF**: dᵢ = ν̂ᵢ − λᵢ·α̂, the intercept shift *beyond* what the severity difference explains.

Effect sizes integrate the expected-score curves E[Y|t] = Σₖ Φ(λt + dᵢ − τₖ) vs the invariance-implied Σₖ Φ(λt − τₖ) over the enriched group's latent distribution N(α̂, ψ̂): signed item bias, dMACS, and the **sum-score measurement bias** — the score distortion pooled-group analyses would suffer *on top of* the real severity gap. Caveat: configural failed, so the metric model is itself an approximation; treat these as order-of-magnitude quantifications.

In [ ]:
push_model_to_r('phq_sum', PHQ_ITEMS, data_genpop_enriched, path_to_helpfile)
GE = dict(WU_ESTABROOK_LEVELS)
fit_measeq_level(PHQ_ITEMS, GE['metric'], r_fit_name='fit_metric_es')
fit_measeq_level(PHQ_ITEMS, GE['scalar'], r_fit_name='fit_scalar_es')
group_labels = list(ro.r('lavaan::lavInspect(fit_metric_es, "group.label")'))
print('groups (1, 2):', group_labels)
with localconverter(ro.default_converter + pandas2ri.converter):
    pt_m = ro.conversion.rpy2py(ro.r('lavaan::parameterTable(fit_metric_es)'))
    pt_s = ro.conversion.rpy2py(ro.r('lavaan::parameterTable(fit_scalar_es)'))

FACTOR = 'phq_sum'
lam = {r.rhs: r.est for r in pt_m[(pt_m.op == '=~') & (pt_m.group == 1)].itertuples()}
thr = {i: list(pt_m[(pt_m.op == '|') & (pt_m.lhs == i) & (pt_m.group == 1)]
               .sort_values('rhs').est) for i in PHQ_ITEMS}
nu2 = {r.lhs: r.est for r in pt_m[(pt_m.op == '~1') & (pt_m.group == 2)
                                  & pt_m.lhs.isin(PHQ_ITEMS)].itertuples()}
alpha2 = float(pt_s[(pt_s.op == '~1') & (pt_s.lhs == FACTOR)
                    & (pt_s.group == 2)].est.iloc[0])
psi2 = float(pt_s[(pt_s.op == '~~') & (pt_s.lhs == FACTOR)
                  & (pt_s.rhs == FACTOR) & (pt_s.group == 2)].est.iloc[0])
print(f"scalar-model severity difference: alpha2 = {alpha2:+.3f} "
      f"(latent SD units of group 1), psi2 = {psi2:.3f}")

# non-uniform DIF: the intercept shift beyond the severity difference
d = {i: nu2[i] - lam[i] * alpha2 for i in PHQ_ITEMS}

t = np.linspace(alpha2 - 4 * np.sqrt(psi2), alpha2 + 4 * np.sqrt(psi2), 801)
w2 = stats.norm.pdf(t, loc=alpha2, scale=np.sqrt(psi2)); w2 = w2 / w2.sum()

def curve(item, dif):
    return np.sum([stats.norm.cdf(lam[item] * t + dif - tk)
                   for tk in thr[item]], axis=0)

rows = []
for item in PHQ_ITEMS:
    diff = curve(item, d[item]) - curve(item, 0.0)
    pooled_sd = float(pd.concat([data_gp[item], data_en[item]]).std())
    rows.append(dict(item=item, nu2=round(nu2[item], 3),
                     dif_beyond_severity=round(d[item], 3),
                     signed_bias=float(np.sum(diff * w2)),
                     dMACS=float(np.sqrt(np.sum(diff ** 2 * w2)) / pooled_sd)))
es = pd.DataFrame(rows)
es['signed_bias'] = es.signed_bias.round(3)
es['dMACS'] = es.dMACS.round(3)
es.to_csv(cfa_dir / 'phq8_effect_sizes.csv', index=False)
print(es.to_string(index=False))
total_dif = es.signed_bias.sum()
# context: the score difference a fully invariant PHQ would show for the
# REAL severity gap
E_inv_sum = np.sum([curve(i, 0.0) for i in PHQ_ITEMS], axis=0)
w1 = stats.norm.pdf(t, loc=0, scale=1); w1 = w1 / w1.sum()
true_gap = float(np.sum(E_inv_sum * w2) - np.sum(E_inv_sum * w1))
print(f"\nexpected sum-score difference from the REAL severity gap alone: "
      f"{true_gap:+.2f} points")
print(f"additional sum-score MEASUREMENT BIAS from non-uniform DIF: "
      f"{total_dif:+.2f} points ({group_labels[1]} minus invariance-implied; "
      f"0-24 scale)")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
E_dif_sum = np.sum([curve(i, d[i]) for i in PHQ_ITEMS], axis=0)
axes[0].plot(t, E_inv_sum, label='invariance-implied')
axes[0].plot(t, E_dif_sum, label=f'{group_labels[1]} actual (with DIF)')
axes[0].set_xlabel('latent severity t (common scale)')
axes[0].set_ylabel('expected PHQ-8 sum'); axes[0].legend()
axes[0].set_title('expected sum score')
axes[1].plot(t, E_dif_sum - E_inv_sum, color='crimson')
axes[1].axhline(0, color='gray', lw=0.5)
axes[1].set_xlabel('latent severity t')
axes[1].set_ylabel('measurement bias (points)')
axes[1].set_title('sum-score DIF bias across severity')
fig.tight_layout()


## D. Exhaustive scalar-target search for a PHQ-8 invariant subset (gp_en)

The stepwise search is greedy and only trims within its metric core; this exhaustive search covers **all** subsets down to `exh_min_items`, capped above by the stepwise run's iteration-0 configural certificate (seed-identical tests exhaustively refuted every size-7 subset, so the search starts at size 6). Early abandon and the calibrated parametric delta screen apply; progress persists per combination and resumes.

In [ ]:
cap = stepwise_configural_cap(dat_dir / 'cfa_other_gp_en', 'phq_sum',
                              len(PHQ_ITEMS))
print('configural certificate cap from the gp_en stepwise run:', cap)
successes, success_size = exhaustive_search_scale(
    whichscale='phq_sum',
    whichcfa='scalar',
    orig_items=OTHER_SCALES,
    datasets={'gp_en': data_genpop_enriched},
    temp_path=path_to_helpfile,
    num_iter=num_iter,
    cpus_to_use=cpus_to_use,
    min_items=exh_min_items,
    max_size=cap,
    alpha_screen=0.005,
    progress_path=cfa_dir / 'exhaustive_progress_phq_sum.pkl',
)
if successes:
    print(f"\nlargest scalar-invariant PHQ-8 subset(s): size {success_size}")
    for com, pair_ps in successes.items():
        print(' ', com, pair_ps)
else:
    print("\nno scalar-invariant PHQ-8 subset of size >= "
          f"{exh_min_items} exists on gp_en (exhaustively verified, "
          "given the certificate cap)")

## Reading the results

- **A** tells you whether the PHQ non-invariance is visible under the field's usual ΔCFI/ΔRMSEA yardstick, or only under the exact permutation test — that positioning decides how the finding is framed against the literature.
- **B** names the structural difference behind the configural failure (e.g., a doublet or a second factor present in one group only).
- **C** converts the non-invariance into score points: the sum-score bias at equal η is the number to quote when discussing pooled clinical+HV designs (brain–PHQ correlations etc.).
- **D** upgrades "the greedy search found no scalar core" to an exhaustive statement either way — and if a scalar core *does* exist, it's the recommendation for cross-sample use.